# 04 Small CNN trained from scratch: RGB versus 13 bands

**Purpose.** Train a convolutional neural network from scratch on RGB (3 channels) and on all 13 bands, with the same split, preprocessing, augmentation, settings and seeds. The only difference between the two inputs is the number of channels in the first layer, so a difference in the results comes from the bands.

**Functions.** `src/piece_of_crop/cnn.py` (`SmallCNN`), `train.py` (`seed_all`, `train_model`, `predict`, `predictions_table`) and `data.py` (`make_loaders`). The same training loop is used for the ResNet in notebook 05.

**Reads.** `results/split.csv`, `results/band_stats.csv` and the patch cache `data/eurosat_cache.npy`.

**Saves.** `results/cnn_results.csv`, `results/cnn_history.csv`, six test prediction files `results/predictions/cnn_<input>_seed<n>.csv`, the training curves `results/figures/cnn_<input>_training.png` and the trained weights in `models/` (not in Git; notebook 07 uses them).

## 1. Setup and shared data from notebook 02

All functions come from the installed `piece_of_crop` package. On a Mac the network trains on the GPU through MPS if PyTorch supports it, otherwise on the CPU.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import  torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, f1_score
from IPython.display import  display

from piece_of_crop import CLASS_NAMES,read_split, get_cache, make_loaders
from piece_of_crop import SmallCNN, seed_all, train_model, predict, predictions_table

# hyperparamters
SEEDS = (0,1,2)
BATCH_SIZE=128
EPOCHS = 30  # fixed, the cosine schedule runs over all of them
LEARNING_RATE= 1e-3

WEIGHT_DECAY=  1e-4


project_root= Path.cwd().parents[0]
data_dir = project_root/ "data" /'EuroSAT_MS'
results_dir = project_root/"results"
figures_dir= results_dir /"figures"
predictions_dir  = results_dir / "predictions"
models_dir=project_root/ "models"
#create the output folders if they dont excist
folders = [results_dir,figures_dir,predictions_dir , models_dir]
for folder in folders :
    folder.mkdir(parents = True , exist_ok=True)

split = read_split( results_dir /  "split.csv")
#print(split.head())
band_stats=pd.read_csv(results_dir/"band_stats.csv" )
cache = get_cache(split,data_dir , project_root / "data"/ "eurosat_cache.npy")

# check so labels goes from 0 to 9
label_ids =np.arange(len( CLASS_NAMES ))
assert np.array_equal(np.sort( split["label"].unique()),label_ids) , "Expected labels 0..9 , something wrong w the split"
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

print ("Project root:", project_root)
print("Device:",device)
print('Patches:' , split["split"].value_counts().to_dict( ))
print( "Cache shape:",cache.shape)

## 2. The network

The architecture follows the CNN of the course notebook (CNN_Class_Competition_2_Food_Classification_Explained.pdf, section 2): three blocks of a 3 x 3 convolution, BatchNorm, ReLU and 2 x 2 max pooling with 32, 64 and 128 filters, then global average pooling and one linear layer with 10 outputs.

- **Convolutions** slide a small filter over the patch, so the same pattern is recognised anywhere in it (lecture_8_DL_for_CV_1_compressed.pdf, slide 27). With `padding=1` the size stays the same, and every pooling halves it: 64 x 64 becomes 32, 16 and 8.
- **BatchNorm** normalises the activations inside the network, which gives "more stable gradients" and "a slight regularization effect" (lecture_6ses.8_intro_to_DL-2.pdf, slide 37).
- **Global average pooling** averages each of the 128 feature maps into one number, so the head has only 128 x 10 weights plus 10 biases.
- **The output is logits.** No softmax is added, because `CrossEntropyLoss` applies it internally (lecture_5_intro_to_DL.pdf, slide 25).

The only change between RGB and 13 bands is the number of input channels of the first convolution: 3 or 13.

In [ ]:
#quick check that both models works and gives 10 outputs
for input_type,channels in (("rgb",3) , ("13band",  13)):
    test_model =SmallCNN(in_channels=channels)
    assert test_model(torch.zeros(2,channels, 64,64)).shape==(2, 10)
    n_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
    print(f"{input_type} : {n_params:,} trainable paramters")

print (test_model)

## 3. Training settings

| Setting | Value | Why |
|---|---|---|
| Loss | cross-entropy on the logits | the loss for multi-class classification; it expects logits (lecture_5_intro_to_DL.pdf, slide 25; CNN course notebook, section 4) |
| Optimizer | AdamW, learning rate 0.001 | the course trains its CNN with Adam at 0.001 (CNN course notebook, section 4). AdamW is Adam with weight decay that "does not accumulate in the momentum nor variance" (external: PyTorch documentation, `torch.optim.AdamW`) |
| Weight decay | 0.0001 | L2 regularisation, "called weight decay in DL", shrinks the weights and makes the model less sensitive to single inputs (lecture_2_introduction_to_ML.pdf, slide 40) |
| Batch size | 128 | as in the course CNN notebook (section 1); the same for RGB and 13 bands |
| Epochs | 30, fixed | the learning rate follows a cosine schedule over all 30 epochs, as in the course's fine-tuning notebook (lecture_12_finetuning_live_WITH_RESULTS 1.pdf, section 4) |
| Learning-rate schedule | cosine, from 0.001 down to 0 over all batches | large steps early, small steps at the end, so the weights settle instead of jumping around (same notebook) |
| Best epoch | lowest validation loss, kept at the end | the validation set chooses the final network, and the test set is used only after this choice (lecture_2_introduction_to_ML.pdf, slide 24) |
| Augmentation | random horizontal and vertical flips, training only | see notebook 02, section 6 |

**Why a fixed 30 epochs with a cosine schedule and no early stopping.** We decided between early stopping (lecture_2_introduction_to_ML.pdf, slide 36; contrastive_learning_notebook.pdf, page 17) and the course's fine-tuning recipe. In an earlier run with early stopping (at most 10 epochs, patience 3), four of six networks were still improving when the epochs ran out, and one was stopped after 5 epochs because the validation loss jumped up once; it ended with a validation loss of 0.356 against 0.205 and 0.223 for the other two seeds of the same input. The spread between seeds then measured when training happened to stop, not the input. With a fixed number of epochs every network gets the same training, and keeping the epoch with the lowest validation loss still protects against overfitting in the last epochs.

The loop is `train_model` in `src/piece_of_crop/train.py`. Each step follows the course order: clear the gradients, forward pass, loss, backward pass, update the weights, then one step of the learning-rate schedule (CNN course notebook, section 4, "Training order").

## 4. Train RGB and 13-band networks with three seeds

For every seed the data loaders are created again, so the shuffling and augmentation follow that seed. After training, each network is evaluated on validation and test with the weights of its best epoch. The training accuracy is the one measured in the training loop at that epoch. The test predictions use the team format from `predictions_table`, the same as notebook 03.

In [ ]:
history_runs=[]
result_rows = [ ]
test_files = split.loc[split["split"].eq("test"),"file"]
test_labels = split.loc[split["split"].eq("test"),"label"].to_numpy()

#train every model, 2 inputs x 3 seeds
for input_type,n_channels in (("rgb",3) , ("13band",13)):
    for seed in SEEDS :
        print(f"\n=== CNN {input_type} , seed {seed} ===")
        seed_all( seed)
        loaders = make_loaders(split, cache,band_stats, input_type, batch_size = BATCH_SIZE)
        sample_images,sample_labels = next(iter(loaders["train"]))
        assert sample_images.shape[1:]==(n_channels, 64,64)  # check the input shape is rigth

        model = SmallCNN(in_channels=n_channels, num_classes = len(CLASS_NAMES)).to(device)
        optimizer =torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
        # the model ends with the weights of the epoch with the lowest val loss
        history, best_epoch = train_model(model, loaders["train"], loaders["val"], optimizer, EPOCHS, device)

        run = pd.DataFrame(history)
        run["epoch"] = np.arange(1, EPOCHS + 1)
        run["input"], run["seed"] = input_type, seed
        history_runs.append(run)
        torch.save(model.state_dict(), models_dir / f"cnn_{input_type}_seed{seed}.pt")

        # train accuracy comes from the training loop at the best epoch
        result_rows.append({"model": "SmallCNN","input": input_type, "seed": seed, "split":"train",
            "accuracy": history["train_acc"][best_epoch - 1], "best_epoch": best_epoch})
        # best epoch is picked on validation, test set is only used from here
        for subset in ("val","test"):
            y_true, proba = predict(model, loaders[subset], device)
            y_pred = proba.argmax(axis=1)
            result_rows.append({"model": "SmallCNN","input": input_type, "seed": seed, "split":subset,
                "accuracy": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro"),
                "best_epoch": best_epoch})
            #save the test predictions to csv
            if subset=="test":
                assert np.array_equal(y_true, test_labels)  # predictions are in split order
                predictions_table(test_files, y_true, proba).to_csv(
                    predictions_dir/ f"cnn_{input_type}_seed{seed}.csv", index=False)


cnn_history=pd.concat(history_runs, ignore_index=True)
cnn_results = pd.DataFrame( result_rows)

cnn_history.to_csv(results_dir / "cnn_history.csv", index= False)
cnn_results.to_csv(results_dir/"cnn_results.csv",index=False)


## 5. Training curves

Training and validation loss per epoch for each seed (left) and validation accuracy (right). The training loss is measured on augmented patches while the network is still changing during the epoch, so it is not directly comparable with the validation loss; so the training accuracy in section 6 is a rough check for overfitting, not an exact score.

In [ ]:
#plot the loss and accuracy for every epoch
for input_type in ("rgb","13band") :
    fig,axes = plt.subplots(1,2, figsize=(11,4))
    for seed in SEEDS:
        run = cnn_history[(cnn_history["input"]==input_type) & (cnn_history["seed"] == seed )]
        axes[0].plot(run["epoch"],run["train_loss"], ls="--",alpha=0.6, label=f"train seed {seed}")
        axes[0].plot(run["epoch"], run["val_loss"],label=f"val seed {seed}")
        axes[1].plot( run["epoch"], run["val_acc"], label = f"seed {seed}")

    axes[0].set(title=f"CNN {input_type}: loss per epoch",xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[1].set(title=f"CNN {input_type}: validation accuracy", xlabel="Epoch",ylabel="Accuracy", ylim=(0,1))
    for ax in axes :
        ax.grid(alpha =0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    # save figure so we can use it in the report
    fig.savefig(figures_dir/ f"cnn_{input_type}_training.png", dpi=150,bbox_inches="tight")
    plt.show()


print("Best epoch (lowest validation loss) for each network :")
display(cnn_results[cnn_results["split"].eq("test")][["input","seed","best_epoch"]])

## 6. Results: RGB versus 13 bands

Mean and standard deviation over the three seeds, for training, validation and test. The per-class reports show seed 0 only; the other seeds are in the prediction files.

In [ ]:
# mean and std over the 3 seeds
summary = cnn_results.groupby(["input","split"],sort=False)[["accuracy","macro_f1"]]
display(summary.agg(["mean", "std"]).round(4) )

#per class report, only for seed 0
for input_type in ("rgb","13band"):
    pred = pd.read_csv(predictions_dir/ f"cnn_{input_type}_seed0.csv")
    print(f"\nPer class test report : {input_type}, seed 0")
    print(classification_report(pred["y_true"],pred["y_pred"], labels=label_ids,
          target_names = list(CLASS_NAMES),zero_division=0))